<a href="https://colab.research.google.com/github/brendonhuynhbp-hub/gt-markets/blob/main/notebooks/app/01_prepare_app_data_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

# =========================
# BLOCK 1 — Build artefacts
# =========================

from pathlib import Path
import pandas as pd
import numpy as np
import re, os, glob

# Mount Drive
from google.colab import drive
drive.mount("/content/drive")

# Paths
DATA_DIR        = Path("/content/drive/MyDrive/gt-markets/data/processed")
DRIVE_ARTEFACTS = Path("/content/drive/MyDrive/gt-markets/AppDemo/artefacts")   # signals_KW_* live here
REPO_ARTEFACTS  = Path("/content/gt-markets/AppDemo/artefacts")                 # write outputs here (cloned repo)

ASSETS = ["GOLD", "BTC", "OIL", "USDCNY"]
FAST, SLOW = 10, 20   # crossover windows

# Input file
MERGED_FILE = DATA_DIR / "merged_financial_trends_engineered_2025-09-07.csv"
assert MERGED_FILE.exists(), f"Missing input file: {MERGED_FILE}"

# Ensure folders
REPO_ARTEFACTS.mkdir(parents=True, exist_ok=True)
for a in ASSETS:
    (REPO_ARTEFACTS / a).mkdir(parents=True, exist_ok=True)

print(f"Input: {MERGED_FILE}")
print(f"Drive artefacts: {DRIVE_ARTEFACTS}")
print(f"Repo artefacts: {REPO_ARTEFACTS}")

# Load merged
merged = pd.read_csv(MERGED_FILE)

# Date handling: use 'Date' if present, else try to derive
if "Date" in merged.columns:
    merged["Date"] = pd.to_datetime(merged["Date"], errors="coerce")
else:
    # try a few common names or the index
    for cand in ["date", "timestamp", "ds"]:
        if cand in merged.columns:
            merged["Date"] = pd.to_datetime(merged[cand], errors="coerce")
            break
    if "Date" not in merged.columns:
        merged = merged.reset_index().rename(columns={"index": "Date"})
        merged["Date"] = pd.to_datetime(merged["Date"], errors="coerce")

merged = merged.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

print("Merged shape:", merged.shape)
print("Sample cols:", list(merged.columns[:12]))

# Close column hints
CLOSE_HINTS = {
    "GOLD":   ["GC=F Close", "Gold Close", "GOLD Close", "XAUUSD Close"],
    "BTC":    ["BTC-USD Close", "BTC Close", "Bitcoin Close"],
    "OIL":    ["CL=F Close", "OIL Close", "WTI Close", "Brent Close"],
    "USDCNY": ["USDCNY=X Close", "USD/CNY Close", "USDCNY Close"],
}

def get_close_series(asset: str) -> pd.Series:
    cols = merged.columns.tolist()
    for cand in CLOSE_HINTS.get(asset, []):
        if cand in cols:
            s = merged[["Date", cand]].rename(columns={cand: "Close"}).set_index("Date").sort_index()["Close"].astype(float)
            return s
    token = asset.upper()
    fallback = [c for c in cols if c.endswith(" Close") and token in c.upper()]
    if fallback:
        cand = fallback[0]
        return merged[["Date", cand]].rename(columns={cand: "Close"}).set_index("Date").sort_index()["Close"].astype(float)
    generic = [c for c in cols if c.endswith(" Close")]
    assert generic, f"No '* Close' columns found for {asset}"
    cand = generic[0]
    return merged[["Date", cand]].rename(columns={cand: "Close"}).set_index("Date").sort_index()["Close"].astype(float)

def ma_series(close: pd.Series, kind: str, span: int) -> pd.Series:
    if kind == "SMA":
        return close.rolling(span).mean()
    elif kind == "EMA":
        return close.ewm(span=span, adjust=False).mean()
    raise ValueError("kind must be SMA or EMA")

def baseline_entry(close: pd.Series, label: str, fast: int, slow: int) -> pd.Series:
    use = "SMA" if label == "BASE_SMA" else "EMA"
    ma_f = ma_series(close, use, fast)
    ma_s = ma_series(close, use, slow)
    return (ma_f > ma_s).astype(int)

def backtest_long_only(close: pd.Series, entry: pd.Series) -> dict:
    # fix FutureWarning by setting fill_method=None
    ret = close.pct_change(fill_method=None).fillna(0.0)
    pos = entry.astype(int)
    strat = pos.shift(1).fillna(0) * ret
    eq = (1.0 + strat).cumprod()

    n = len(eq)
    if n == 0:
        return dict(Return_Ann=0.0, Sharpe=0.0, MaxDD=0.0, WinRate=0.0, N_Trades=0)

    ann = (eq.iloc[-1]) ** (252 / n) - 1
    vol = strat.std() * (252 ** 0.5)
    sharpe = (strat.mean() * 252) / vol if vol and vol > 0 else 0.0
    max_dd = (eq / eq.cummax() - 1).min()
    trades = (entry.astype(int).diff().fillna(0) == 1).sum()
    wins = (strat > 0).sum()
    winrate = wins / max(int(trades), 1)
    return dict(Return_Ann=float(ann), Sharpe=float(sharpe), MaxDD=float(max_dd), WinRate=float(winrate), N_Trades=int(trades))

def backtest_long_only_filtered(close: pd.Series, base_entry: pd.Series, kw_signal: pd.Series) -> dict:
    entry = ((base_entry == 1) & (kw_signal == 1)).astype(int)
    return backtest_long_only(close, entry)

# Baseline + per-asset files
summary_rows = []
for asset in ASSETS:
    closeD = get_close_series(asset)
    dfD = closeD.to_frame()
    dfW = closeD.resample("W-FRI").last().dropna().to_frame()

    for freq, dfX in [("D", dfD), ("W", dfW)]:
        rows = []
        for strat in ["BASE_SMA", "BASE_EMA"]:
            entry = baseline_entry(dfX["Close"], strat, FAST, SLOW)
            met = backtest_long_only(dfX["Close"], entry)
            rows.append(dict(type="baseline", asset=asset, freq=freq, strategy=strat, **met))
        out = pd.DataFrame(rows)
        (REPO_ARTEFACTS / asset / f"metrics_baseline_{freq}.csv").write_text(out.to_csv(index=False))
        (REPO_ARTEFACTS / asset / f"leaderboard_{freq}.csv").write_text(out.to_csv(index=False))
        summary_rows.extend(out.to_dict("records"))
        print(f"[ok] {asset} {freq}: baseline rows={len(out)}")

# Root summaries
pd.DataFrame([r for r in summary_rows if r["freq"] == "D"]).to_csv(REPO_ARTEFACTS / "metrics_summary_D.csv", index=False)
pd.DataFrame([r for r in summary_rows if r["freq"] == "W"]).to_csv(REPO_ARTEFACTS / "metrics_summary_W.csv", index=False)
print("Summary rows:", len(summary_rows))

# Keyword metrics — read signals from DRIVE_ARTEFACTS/<asset>
def parse_kw_file(name: str):
    m = re.match(r"signals_KW_([^_]+)(?:_w(\d+))?_(BASE|EXT)_([DW])\.csv$", name)
    if not m:
        m2 = re.match(r"signals_KW_([^_]+).*_([DW])\.csv$", name)
        return (m2.group(1), None, None, m2.group(2)) if m2 else (None, None, None, None)
    model, window, feature_set, freq = m.groups()
    return model, (int(window) if window else None), feature_set, freq

kw_root = {"D": [], "W": []}

for asset in ASSETS:
    asset_drive = DRIVE_ARTEFACTS / asset
    if not asset_drive.exists():
        continue
    sig_files = sorted(asset_drive.glob("signals_KW_*_[DW].csv"))
    if not sig_files:
        continue

    closeD = get_close_series(asset)
    closeW = closeD.resample("W-FRI").last().dropna()

    per_asset = {"D": [], "W": []}

    for f in sig_files:
        model, window, feature_set, freq = parse_kw_file(f.name)
        if freq not in ("D", "W"):
            continue

        sig = pd.read_csv(f)
        # normalise date index
        if "date" in sig.columns:
            sig["date"] = pd.to_datetime(sig["date"], errors="coerce")
            sig = sig.dropna(subset=["date"]).set_index("date").sort_index()
        elif "Date" in sig.columns:
            sig["Date"] = pd.to_datetime(sig["Date"], errors="coerce")
            sig = sig.dropna(subset=["Date"]).set_index("Date").sort_index()
        else:
            first = sig.columns[0]
            sig[first] = pd.to_datetime(sig[first], errors="coerce")
            sig = sig.dropna(subset=[first]).set_index(first).sort_index()

        if "signal" not in sig.columns:
            cand = [c for c in sig.columns if c.lower() == "signal"]
            if not cand:
                print(f"[skip] No 'signal' column in {f.name}")
                continue
            sig["signal"] = sig[cand[0]]

        # Normalise model signal -> binary {0,1}
        raw = pd.to_numeric(sig["signal"], errors="coerce").replace([np.inf, -np.inf], np.nan)
        if raw.dropna().between(0, 1).all():
            bin_sig = (raw >= 0.5).astype(int)
        else:
            bin_sig = (raw > 0).astype(int)
        kw = bin_sig.fillna(0).astype(int)

        close = closeD if freq == "D" else closeW
        kw_aligned = kw.reindex(close.index).fillna(0).astype(int)

        for strat in ["BASE_SMA", "BASE_EMA"]:
            base_entry = baseline_entry(close, strat, FAST, SLOW)
            met = backtest_long_only_filtered(close, base_entry, kw_aligned)
            rec = dict(
                type="keyword",
                asset=asset,
                freq=freq,
                strategy=strat,      # same labels as baseline
                model=model,
                Return_Ann=met["Return_Ann"],
                Sharpe=met["Sharpe"],
                MaxDD=met["MaxDD"],
                WinRate=met["WinRate"],
                N_Trades=met["N_Trades"],
            )
            per_asset[freq].append(rec)
            kw_root[freq].append(rec)

        print(f"[ok] KW metrics -> {asset} {f.name}")

    # write per-asset keyword metrics
    for freq in ["D", "W"]:
        pd.DataFrame(per_asset[freq]).to_csv(REPO_ARTEFACTS / asset / f"metrics_keywords_{freq}.csv", index=False)

# write root keyword metrics
pd.DataFrame(kw_root["D"]).to_csv(REPO_ARTEFACTS / "metrics_keywords_D.csv", index=False)
pd.DataFrame(kw_root["W"]).to_csv(REPO_ARTEFACTS / "metrics_keywords_W.csv", index=False)

# Audits
print("=== REPO ARTEFACTS ===")
print(REPO_ARTEFACTS)

for f in sorted(glob.glob(str(REPO_ARTEFACTS / "*/metrics_keywords_*.csv"))):
    sz = os.stat(f).st_size
    try:
        rows = len(pd.read_csv(f))
    except Exception:
        rows = -1
    print(f"{f}  size={sz}  rows={rows}")

for f in [
    REPO_ARTEFACTS / "metrics_summary_D.csv",
    REPO_ARTEFACTS / "metrics_summary_W.csv",
    REPO_ARTEFACTS / "metrics_keywords_D.csv",
    REPO_ARTEFACTS / "metrics_keywords_W.csv",
]:
    if f.exists():
        try:
            print(f"[root] {f.name} -> {len(pd.read_csv(f))} rows")
        except Exception as e:
            print(f"[root] {f.name} read error: {e}")
    else:
        print(f"[root] {f.name} missing")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Input: /content/drive/MyDrive/gt-markets/data/processed/merged_financial_trends_engineered_2025-09-07.csv
Drive artefacts: /content/drive/MyDrive/gt-markets/AppDemo/artefacts
Repo artefacts: /content/gt-markets/AppDemo/artefacts
Merged shape: (2609, 255)
Sample cols: ['Date', 'BTC-USD Close', 'CL=F Close', 'DXY Close', 'GC=F Close', 'USDCNY=X Close', 'BTC-USD Open', 'CL=F Open', 'DXY Open', 'GC=F Open', 'USDCNY=X Open', 'BTC-USD High']
[ok] GOLD D: baseline rows=2
[ok] GOLD W: baseline rows=2
[ok] BTC D: baseline rows=2
[ok] BTC W: baseline rows=2
[ok] OIL D: baseline rows=2
[ok] OIL W: baseline rows=2
[ok] USDCNY D: baseline rows=2
[ok] USDCNY W: baseline rows=2
Summary rows: 16
[ok] KW metrics -> GOLD signals_KW_GRU_w30_BASE_D.csv
[ok] KW metrics -> GOLD signals_KW_GRU_w30_BASE_W.csv
[ok] KW metrics -> GOLD signals_KW_GRU_w30_EXT_D.csv
[ok] KW metrics -> GO

In [7]:

# =========================
# BLOCK 2 — Commit & push artefacts only (auto-clone with PAT if needed)
# =========================

from pathlib import Path
import subprocess as sp
import os, re, getpass, datetime, sys

OWNER = "brendonhuynhbp-hub"
REPO  = "gt-markets"
CLONE_DIR = Path("/content/gt-markets")
ARTEFACTS_REL = Path("AppDemo/artefacts")

def sh(cmd, cwd=None, check=False, capture=False):
    if capture:
        return sp.run(cmd, cwd=cwd, check=check, text=True, capture_output=True)
    return sp.run(cmd, cwd=cwd, check=check, text=True)

def parse_owner_repo(url: str):
    m = re.match(r"https?://github\.com/([^/]+)/([^/\.]+)(?:\.git)?/?$", url)
    if m: return m.group(1), m.group(2)
    m = re.match(r"git@github\.com:([^/]+)/([^/\.]+)(?:\.git)?$", url)
    if m: return m.group(1), m.group(2)
    raise ValueError(f"Unrecognised origin URL: {url}")

# --- 1) Ensure repo exists: try public clone → fallback to PAT clone ---
if not (CLONE_DIR / ".git").exists():
    print("Repo not found at /content/gt-markets. Attempting clone...")
    CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)

    # Try public (works only if repo is public)
    pub = sh(["git", "clone", f"https://github.com/{OWNER}/{REPO}.git", str(CLONE_DIR)], capture=True)
    if pub.returncode != 0:
        print("Public clone failed (likely private repo). Will use PAT.")
        token = getpass.getpass("GitHub Personal Access Token (PAT) [scope: repo]: ")
        # PAT clone
        pat_clone_url = f"https://{token}@github.com/{OWNER}/{REPO}.git"
        priv = sh(["git", "clone", pat_clone_url, str(CLONE_DIR)], capture=True)
        if priv.returncode != 0:
            print(priv.stdout)
            print(priv.stderr)
            raise RuntimeError("Clone with PAT failed. Check PAT scopes (repo) and SSO access.")
        else:
            print("✅ Cloned with PAT.")
            # Do not keep token around
            del token
    else:
        print("✅ Cloned without auth (public).")
else:
    print("Repo already present at /content/gt-markets.")

os.chdir(CLONE_DIR)
print(f"Repo root: {CLONE_DIR}")

# --- 2) Confirm remote + branch ---
origin = sh(["git", "remote", "get-url", "origin"], capture=True)
if origin.returncode != 0:
    raise RuntimeError("No 'origin' remote set. The clone looks broken.")
origin_url = origin.stdout.strip()
print(f"Remote: {origin_url}")

branch_cp = sh(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture=True)
if branch_cp.returncode != 0:
    raise RuntimeError("Cannot determine current branch.")
branch = branch_cp.stdout.strip()
print(f"Branch: {branch}")

try:
    owner_detected, repo_detected = parse_owner_repo(origin_url)
except Exception:
    owner_detected, repo_detected = OWNER, REPO

# --- 3) Set identity for this repo ---
sh(["git", "config", "user.name", "brendonhuynhbp-hub"], check=True)
sh(["git", "config", "user.email", "brendonhuynh.bp@gmail.com"], check=True)

# --- 4) Fetch + pull (rebase) to avoid non-FF rejects ---
print("\nFetching and rebasing on latest remote…")
fetch = sh(["git", "fetch", "origin"], capture=True)
if fetch.returncode != 0:
    print(fetch.stderr)
    raise RuntimeError("git fetch failed.")

pull = sh(["git", "pull", "--rebase", "origin", branch], capture=True)
if pull.returncode != 0:
    print(pull.stdout)
    print(pull.stderr)
    raise RuntimeError("git pull --rebase failed. Resolve conflicts then re-run.")
else:
    print(pull.stdout.strip() or "Up to date.")

# --- 5) Stage ONLY the artefacts directory ---
artefacts_abs = CLONE_DIR / ARTEFACTS_REL
if not artefacts_abs.exists():
    raise FileNotFoundError(f"Expected artefacts at {artefacts_abs}. Run Block 1 first to write files here.")

print(f"\nStaging: {ARTEFACTS_REL}")
add = sh(["git", "add", "-A", str(ARTEFACTS_REL)], capture=True)
if add.returncode != 0:
    print(add.stderr)
    raise RuntimeError("git add failed.")

# Nothing staged? exit quietly with status
diff_cached = sh(["git", "diff", "--cached", "--name-only"], capture=True)
staged_list = [l for l in diff_cached.stdout.strip().splitlines() if l.strip()]
if not staged_list:
    print("No changes to commit in AppDemo/artefacts.")
    status = sh(["git", "status", "--short"], capture=True)
    print(status.stdout)
    sys.exit(0)

# --- 6) Commit ---
stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
msg = f"AppDemo: update artefacts ({stamp})"
commit = sh(["git", "commit", "-m", msg], capture=True)
if commit.returncode != 0:
    print(commit.stdout)
    print(commit.stderr)
    raise RuntimeError("git commit failed.")
print(commit.stdout.strip())

# --- 7) Push with one-off PAT URL (does not change origin) ---
print("\nGitHub Personal Access Token (PAT) will be used for this push only.")
print("Required scopes: repo (and SSO must be authorised if your org enforces it).")
token = getpass.getpass("GitHub Personal Access Token (PAT): ")

push_url = f"https://{token}@github.com/{owner_detected}/{repo_detected}.git"
print(f"Pushing to https://github.com/{owner_detected}/{repo_detected} on '{branch}' …")
env = os.environ.copy()
env["GIT_ASKPASS"] = "echo"  # suppress interactive prompts

push = sp.run(["git", "push", push_url, branch], text=True, capture_output=True, env=env, cwd=CLONE_DIR)
if push.returncode != 0:
    print(push.stdout)
    print(push.stderr)
    raise RuntimeError("git push failed. If remote changed again, re-run: fetch, pull --rebase, then push.")
else:
    print("✅ Push complete.")
    del token

# --- 8) Final status ---
print("\nOn branch", branch)
status = sh(["git", "status", "--short"], capture=True)
print(status.stdout or "Working tree clean.")

Repo not found at /content/gt-markets. Attempting clone...
Public clone failed (likely private repo). Will use PAT.
GitHub Personal Access Token (PAT) [scope: repo]: ··········

fatal: destination path '/content/gt-markets' already exists and is not an empty directory.



RuntimeError: Clone with PAT failed. Check PAT scopes (repo) and SSO access.